# Step 1 · LTX-2 duration test

Compare 49 and 65 frames at 24 fps using the archived 19B Dev FP8 configuration.
The default three development prompts cover a static close-up, a moving forearm, and a wide walking shot.
Both lengths use the same prompt and seed for each comparison. This plans six clips.

## Prepare Colab

Select an NVIDIA L4 GPU runtime with High RAM. Leave roughly 60 GiB of local scratch for the weights.
In Colab Secrets, enable a Hugging Face read token named `HF_TOKEN` with access to the
[pinned Gemma encoder](https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized).
Upload this notebook by itself. It bundles the runner and exact prompt CSV.
Leave `SOURCE_REPO` set to `None` unless you want to use a separate checkout.
Only job outputs go to `RUNS`. The CUDA environment and checkpoint cache stay on the Colab VM.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
SOURCE_REPO = None  # Optional path to a full checkout.
RUNS = Path("/content/drive/MyDrive/TattooGen/runs")

In [ ]:
# Bundled runner and prompt CSV. Regenerate with notebooks/sync_colab_assets.py.
import base64
import zlib

if SOURCE_REPO is None:
    packed = (
    'eJzVXA1z2zaT/is439xEaijVkr89r2au17itp22SS/z27ib2cCASlBBTBMsPO27G//12FwAJfujDTpqbc+eNJRJY'
    'LBYPdvdZwO/nvbj4NPUTXsg7MU4f9s7Z3jX99zbmCeNJyLIyYcVSMN2G/Xb136MpeyXu2E9vT9lCJCKDFyphH9U8'
    'Z2UuQjZ/oA4/qpjPWaIKMVfqNh9rudfJdSJXqcoKxrNFyrNcVA+C/K76vOT5Mpbz6vvHXNU9VX6dRJlasZQX2IqZ'
    '52/ha9UoL+dppgKR59WjQqzSSMb1iIVcCdToOnl38fbN+8urN+/+h83Y9d5vcrEsMhnc5t/TjFHxH9+8uvDfXfxx'
    '+f7yzWtqJc5Oo+PwbHoSnh4EZ1MRTefimIvg5Dg4PTzcn0/ODvaPxcEZ9v6vi8uff7l63xQQRkEwneyfHpweRGLC'
    'Od8/Oj2ch9HRUXhwfMrhIw/Ds/2Ahv/l4sdf3765fH1FXWHhRtPR5Gw+CsXdKEpPxzmPRCGSXGU5tr94jQq/o8YL'
    'pRax+H4hVis+OhhNpvORLEZ/cvjfob8/KpM/S54U8i8ROj2bqh6fRidCHEbz8Ohs//Tk8OCYh/zkRITT8HgaHZwc'
    'RGF4eHhKlnp38ePl2wvo9vk6YfBzvbcUaNBrwNfRZOrBg3sZwlrB95PjU/wepTl+mx7il7wQ+uvhPn5dlDLkSSDw'
    'ycF437NCeRlK5btvT8b7uv/Cj/mDyEjIh+nZjX2aBzymlhNHTiaqx/vjE2y6UiGPZfFQtz/QkhOxoH3gA7RWKU0I'
    'UV2J0o99kSxRpZVIqMlPPM4Fdk9lKmKZCN1PbylfJcLPC74QPqxiQ5YIZA57S7eeR5NjeAufzGLRvtPvoOOIVA1H'
    'q5UjQkVRrHgok4VuV4hPqFygQpH5ocxv/SLjSR6pbAUPwA4yJJUrAcFSBLepkknh08YBKTUOURcUBR3vKkUbu6SS'
    'Y8fMRKpyWajsAdsaoHmNBrWoNg5B2qPeraGI2IrfwjKAmxqk4CzAvOA+fHQIHosyvhK5x8xqyBA+50KEw3Otj4zQ'
    'L7F99o8ZPWf/YNPvvjs+MK/xJ+MyF+wPHpfiIstUNrjee48tV2VesLlgHCQkFg3s/X/+JgvBwExiITJwdcPGQFof'
    'pjIWi2Sgvw3Zv8zoay4K+2iITXjyMEhAozP8krB/Y6fYcsIi+ioTI224Wdl/wgNY4EImgRmfBapMCtAiIveMa85O'
    'k5cTjwkeLBkvQBsOkzur1b+XxbIy61iloC2tE+Bpdr1XFtEIYCcXiMpE3COwZ7gbhoznDOHiqqjuc3AIMag0AInj'
    'VzIo3gkOSz7AlkMzol4xbPkZjDmAXh+qTSXBOd0Mz1njISIaHo9z8NbpYEhWggZoJxzysVoHNLWRXpkeW5DNcZF4'
    'HNsG4zs0ZT6oFySVCBK9CPgZxFthm9fhCiPh+z80bgKVFBy6lon8sxSMtgLC5/JVTrEWIAURqniwshtAIpSzGfri'
    'OXwybqQeHJoglNw54nfY3wsxgFU+mujp1JuiB5ATR+CaGf2HGZxl4s9Sgs8gOB3ts5AX8Kqo1hBnBBMCJHyCjeNg'
    'sJ5WveSkD2gMUVmE1SR0KxhO76P1qsMG3r4OYWlylULAYiSwnXMjslqBahcU96qxbRpLUWMJdWnsZOdxhazWc+11'
    '6pXarPg7sDKoCxZ0VLVeqNrhqDvEixy8N2Kz0KirlabsbMY+3OivCGMJXvSTZ9EsknKFqZxwNXU0+1f2ThXwGie6'
    'KJY5ZnlGoUKxHMbmsMYyy4sRjMVEFIkAMRBkKs9tp3EtDyITYmWm1QAv56xm3Qr1JPvXfu+D7nl+w17aJ+f60U0L'
    'uzjnMU/Ba4WDzw0vco6z9ljLiZxb5H6AtzpjEIJa429KUmg8fEJKPRpNM1GUWWKTHR3w8qIMH3zasySZQpTbwAwN'
    'euvQbHbP9d7avQ77AB5SciJi6BiDhSlSNuTqt3eQ+5goGl3vfa6z28d//9zORB9bEjDtSElrncc1XqJR8RX+Ni+c'
    'mIyu3MfUfYARw+KHwkgdQmCE+ZogoQ1pcv8xSQvlAuBPQYKWZMmnR5AHDcdL8cm8G7oKJDJfogehREB8SsFIdeC/'
    'g/xGAehIGVTLz8sI/BPoNF6lh+24Tc3EJ9hkEAu6elJap59i4hTwDEQjURlj0pWTDmPcGASwgY1xkRQxObtBCyZe'
    'CxbOd0RtB7Bed7lr1NYzGWAAMwqOF+B4bsUD+SxrnQ/w4Ib2GnygnUYaDpvbCV67QnCcApzPIsdRPz8OzVO7R5oj'
    '1M9vOlLR0rQwY5lTljkgz1ljiV7qgNZQgJ77FSI2OlLYBRefyFsumCqLtKRoHMUSvRQBtFhKCAdIe2FwGBfi2QqS'
    '8pAUyc/ZZ1zOx8qwBgNXWSlq+CG+fZvkV5CxZFNlwdJAAMnrslwsQJ2IBzDPsiKxywi/+aG6TxBGkLkmPM2Xqqge'
    'OTKABfqBysTY5QRjoAOauYT+amXlzksZh36qYMoPLQlW43xcyClYtaYktvPV5fQPGb5JxHt8+tbSmHViykLG+Xge'
    'KyDQFTkn6F7oPH9LT41qNHjV/Xd8hAQ/39K3eEhF1euNJkDYGdfI2du0GuOgDDnCjt9xGfO5xZ7zEqDmA8OWsEgB'
    'T/lcIi8cYMYxAOJ61gXdO4gMctVIPAodwE0JBQsncx7cCqyu2BwKYPfbIRu8//30TOcN4l7AhivTlDIi9vPbf9bR'
    'XOsDHkTrqb/CWKhw1QhAULWA9cwHg8mxxybHQ8/0n+lf8BVNNtMt52ivAlqNCzXQj+jJqS8OVwdRYoQTulri24JN'
    'U92kxuMAFPNQu3GBXei5z2f0236d26+wVf1e9VzZtE75QxIsM5XIv4QOCPo9WBnUVPlYJHcSXhvX8ctP/tWbXy9e'
    'tz0+ddickF0kiBNKsawUU+vKRQBegRwpvgVuiAuHNRdm6G29hGmG/OZ670ck2OiUdDseYMHKpnMpz/BVKoFphqzm'
    '4tolod+N4jJfztAJGbkt7zFwKDb6O7kYUyUNulqmPevQbG2FGf1rxDpjzzpj1NmF16gOVCO0M46+EYyFkAO0/d2g'
    'Xo9qOrtp77EV/+Tfq+wWAuRs6mQywPbUPXqYQmRJPoMA9V1lGfxMHsh+aVTW9KOPMvkIe+3GiLSrSh5r5jirMfop'
    'f6US8LvFclDb0WNUjfMzpYqZmbyVYjwaCOr3u45BHE85o389FquMw4xu2vu87uQGi5kbFxz9YGua6pGPY8wcTzp+'
    '/eb1RWviRrNxVQCzq9lw+47iRtm+zd1xUGsVeXX5/temIiYsV4W2KjRn4G5BAfvCwwzWY26auiZMVyE2gImAp0My'
    'iDVHQFQVm8q4kL9jzfBnevGWQ7LTjlMkg5ZrrBOXO16F1x/+efXGv7r87fL1z4iLwtctgmWZ3OY+ULP59pApQsl9'
    'qaxIvQJakHWHaAet+oD8a09O2z8VZ93IG0Smklp5bVtanWnK8KFRbb3B7aode6tRVXm98ZojNAuwdYd2YRZE57cy'
    '9bFiPNvXiui0o6mJrQa74zhRoooiSKELPxX81l+JlcoeMA3C7N+EvYJn6AExwI9TkUU+EUGwp1sx0/JkEgkAHeQN'
    'uOgt/pCDmZGHdDe0Dg64Z2aA0E6hSxcyzSud6d9g9a1Rlq7n3qlXtw2tC/N1B1uoB6FUoq/fmIo9jlaufJ3OGz2q'
    '3L4lnJ77WFWoxVCh3wipTUQVf3fJRNojT+8KjWDweYjMmcFz1bU+DrgZtrrX5wXru7fPFEAI7CjwveRRm/JMld8a'
    '3VT5YTdC0PZ1tJ01djYGozkvAqCdkKbMJk0w2o8yMgAZ14ZG8tOydW+lrpF8AuexEcNsb8giPndkP9r6NNVOq3LT'
    'ZxzvhX714uaxWbRzvcugZ5FmZhT6gtle3kUAmdo2pC+eoWYUz2Z5kekyQi8KGr5xtsZnDjqT9axtG8s07F+J9eml'
    '3sXtss9335mKSaNW1N4jeHIVFCWP/bpNV0+36FIfs/qQGfrAqsHD+5BuqiSk7j3+iI20t2oKSktq3ktvEhh3MGwW'
    'qdARYqoUYC7rL+S81R0RbRxl1QzG/h7PUw72G7J0XMVqGqyrIQW2cOH7w24xKSsTc74D/1hoeFREl0mJ52a+QJxb'
    'z6objFe3oUTgZBipKTXGOhBQf1/dupkyRNK/NDnQ5YDvcb4wkkkDK1qg2/WVgeClU+0x7dx6D+5aFLm9qH6FxQej'
    'SKRiTJ2wBMFZKCPykYXVl1RkPy6VyvEUKgLkLE2XeotipdAZVEclc/g9vhKYIfDs4RUwzwDP4gZgsZkencpyhW3R'
    'UpxKAiEmdbAtB1UrXG8ze8RQX5/xfSYLoe1CNgvLVZqblcUScFLMpkP2Ek9TYf33hr1CgBHFHJiuHsymnpDYIUuq'
    'q9umekqm+tCqsTkFcKxTy8S20oVN16uCF1yBFKyfUudq+Bf+C4+9GL0YPo7w+sQo1a6yKtW9uDnfP3gc5foxBmnw'
    'n6Oo5VAtxGioqnxYgRGHRQ0eG3ikxTVFtfp8vfY/lNSuqT53is5m5u7DdojrqSebXq3nbsfHxhapSrJmkt2qrP2x'
    'nPgPkclIIuMFxWKBU4V5IfNCg/SQ3h7IO8iwZwA9CrTKvbq9I8Vq9EMc6yMUq1EzGupIYNNJMGSJGemWVNFhea2y'
    'YUOMdvL9gtDH14MZZHMZl1hRmrH9GuvV1C3o1071ZxNrEjrWtQXXDYZHCk342TZh/Ck6PuVr+Sb8seX9rnuy8+it'
    '+fd4LbMAVN8Ggf3kUdelux0NnHRNG/nDYAi/KOfrUZoA1Fc5NJyZ3WeqEFgg1GfDWmzHRboaEzXIKbA2UATblM2a'
    'wOoKAburrMe52J/KyaxEsVShuZxSnVLVhxJkVXprrlB5/QLbhfzzTulfn2rY04bzap49Ah/XTWicqtQ9nui2g/lw'
    'PIWDqWsbt8Ci/fCGjmuCnB5/lzCHP3poG2o24naTKi0BrqP8FIi0YBf0C0/DYVNRHtXCZuVIXs7YpGfPDpy8ybYl'
    'G8W4pcxJH0dAtO+M9B/+VeOiHyBLukZcA0aK2wSJhqO63qMZ2WxTp4n4mC782YQZ/xm02UUtuXJs2/Lshgfsw+T6'
    'FTeuopPT9tiFfERfwIRM4SewP1I21wqP5+wzyXrsLRQTFHpipgkD7/mdCN0IgLu5RwwGeLP4m09CMJ+xLR91MI1I'
    '6zF7LwRrIkjfPBV091RheM3oTulKu736BphMqoScrpJiBmWvlY5/yBYl3sB7S28GociDTBLkZ74fqgBIh9t1zMPQ'
    '56YPWGA0qo5lzSlNaNhEsFTAl/IZNLJXSnw84tHOr3aFwy3y8b5rVzoVRDGCbemtt98XCLCu0GMJPM9hd77EL9Sd'
    'CtQNudtsRUnkSIbrBW6RoItZ7viwxhxY8exwuqUr4AR7croPMcP0V2Xgh/Fodm/bGpidN1LJyPiMzYJwZnSDgOTR'
    'L5RYlQjpBHnm3E7El2OTgdNnWHbzyZYkdBPnsiI9oBuL1S6jRzBRd5c1CTK1sCxZj9Shyr30UO/59bTMbDlQwqeg'
    '7vv6aorv4w70/eoamt6Q1wmYsH1LxbdX2mD2eMu8mq23oNzKN7eXaDd5eBmWLq55+S2MUCjIueYKKA2Cwyshn8/8'
    'IFbFEqKKF6t79yu1o1IESsqLh1h4K1XIyAtUrDKqxnqYi3kwWKGUD4N5KbBpL+B4BQrvxRZY17PfQQ8QEWNdFOWb'
    'g0REkKeCABwijoMjwK9IBWXu4dGUPij23OptMvFWPA9KSiEnp6PpmQf8CZRF0R5eZReFDECt5BamnHr5UmVFbqbL'
    's5VXZDyUOA6PvYw0xhl5WPovVx6VAjzUHYOtBwYBaJYgBjy4N89oECSGUnl40RBbkqzswVjCi8o49ioKgOUFnVEr'
    'gB+rVB+ZpULS/QBhcMF4iOVsygpwQowmhIV5/ScE+JcEZnL6FoLH7oU+2uQMZwtOPqWWesZj9oOua3hMT22EywW9'
    'UL8RzZk5pmBoCqanwGTOTBmKKX0PjswH+2E1ZngWn5fzjwBJbGhNBR8kEDzUlmk7xTgHmAFZi90vpTnw1YiAX2lB'
    'qRF8RCuPypTiJJhL3DPCQD0+Ko1ZO0AitmriH0MkUy8SKyDHsBQH+6ODMw0xL4/lSi+9n8dC3AE4eQIwABonEATw'
    'byzzlXev4sjDg0WsS0JsjmHLC28p8Jgd1pwy5Boa4Alg1bMVIAf2tQLUgOsqJHyF4eawKe7otrpnzFO9XYcGq7oL'
    'BgcGbQQwnFV38WmaIz1NXH2ZmauMOGOEgZkO05Nj9XSZMYNdd1IKTdKDAGO57vprIzFtEH3qDt4IVx/sgW+wtx4A'
    'pIGILhR0co9IMFC13oORzQkRY/arECmGjMLevjZaJxxwxBEfkA9BmKGyzhySarqHLyOJVw08bVKCFfrqFVgPe+jJ'
    'mfkQog48mEKmFg+JAi90uD86PLMIgHQuQ0fXdi7wq8TaIbhhsCr+yQv4qCTkMTfOJQfIxOiQQ++ex3hVgdyEj27W'
    'U/MYbzJ7IUAWIg/hinyLbWrhtMmzEESs1mvwpGfRdCl6Rjt7FD0RRjNxPUk1cWYmvt6VWGM1kYRzzd2l1IbI0aGA'
    'YZiNHAY8wG9WdB9arXAfoSJoS2ZsySjcGMg4UCkTByI4MzRSJmrXshkmCq0FTs6Fy6ETjo72R0dniA7jRyqcaOcT'
    'LCG/9bAlXqbw8gTSG9f59IeghvdpRiDrXyhaZ8rEfROHtjmf3lCkq+YOZkAL1wPRvLagpfI7veHH8T7WEowssR4x'
    '1u3rIERWHLPLNYHG0yCropLB1wJCXUnIIjvpC/eSLqnlPcFpF4/UjlEmKnVAZxZis38iLB3Vwex4/6UXckDJ2owG'
    'b7958xj+JTRlsJWTraGsymeodzOWPSef6Ytg8FuqBohwIl+WyawPYZUFGFlgPYrwwA9XS6MI59/0QGCivEpitO/Z'
    'kLMkTV9SZTDkkeYCkmjQ9haQYCOg9jRWPQtpnCXeFUYkEQSOG9HHyW8JDJTWxCpZ2KzGYAGS9sUS4kUhMh1wbpUE'
    'Hwm7b4f4Q6mtG3ys52jHny0eZW0I6s9vW6BYk9zgZHtym96IxGoDMGuACg8J0y+6wCDj/S3RqEpnn+AUDALXpS4b'
    'ItGJE4l0LuwSI5O6NHJig56AQxLs0iI8oOF5vpEZmac2eemhRs2c+NnsaAdetC6JWZcZP48hGaus9zBoxycHphyY'
    't86VzR+BValyLBYbAlPLAbXDlE2EnkSmTuv4o/PedaRa5zM1p7Z0KtYsfscYRI6nn091w9BX5lI7x6Bnsii0xO48'
    '+mmxZwe+3MhFXM4Uilwukn7OpIeMBSfaJnsSlLNGdNLprnEE2iw9sclS7pob4V/4Zni/kGJjJilZz3cJVToJ/nuI'
    'Uif17dClddlvT4wyfwzj7Gb9HrLPOKYk5MnEqkZXv/125exfKbZ1mPmzadaGkDbZd2IapsTIrSgJarmiioFX7GqF'
    'Z0v4d8K6BLqVZlVeqT+ifX2q1U2TW1yrPyH6Qqrl+qimiTZS9ecFtSbVItjs5NY2Ba8OyzIebXeWNZnUYU4n2Jpo'
    'bcqQiLnXZCtdKli9T19SOfxabKuTXHfJ1lPSI/3n9AovuRbKcCYncpEhnsHNjME25E6a1T81KO6aDq1n7WuTocm0'
    'EeycjHoNL9eeiIi1w8SW5QoWbTGXWbhLhKPufycZ20LDnp4XrSFgzrx34GDbyPkXBC1NxvH/lKAOXhYX64vKu5aS'
    'NwWvAyd46XzaIWRr8yXN5V06JjienX3RMdU34mJPyJCez8PIHOs9iaHzT60PVkcWO9Gw7YWgpx9lTQ7ruKRT63Vn'
    'WS5vt9QLAu6u4chmOFup15ecY3XS6S88zdol3vQQsYSBYdZy9WeHm418+5kl4KefPUyOGiEKE+R1J1Vtxl6zsZwH'
    'mQgt0Xhg5s85/6/pWDc9fvbxVV+oaniXtgmeegz+/4pSHXfuTyCpqtKZDQy+4lb3YO2IbozkTyBWPbHp67OqTjrc'
    'JFWb0puvQeN35mG1CdejraLv35Z97VZUeh71Omlf16BcWnvynusaFa2vqVcIG36HIuPGUPfMlGhLibF9oLEmF3oS'
    '66pxZyyxa02ypl7aYE89j6do2H+no50gbb7S8XfUqSenPXc0NpySucfvDjX7KED+rqdk3Vj3jZnZ7gdkvWhyOU6b'
    'y2/jcpWhdmByPUT+G4THpydYm0LkWedOR89JWu95vEvbQr5Y2A0rVPKwG4PTx/pdBtcbIp/O4Dpp+VPP1L7CVUNt'
    'mBEIGJFhnnZm/y2C4d91uj/db97waJ2v9URAXQ2wJK+QYLhdY1/PFY+vf12xm6Y/5aRt49VFEPqxpL+i7/diVQAy'
    'jP+pR3RkzN3uAzwhGm4+te3EP8zwvi1p3Hv8X2QmSYo='
    )
    assets = json.loads(zlib.decompress(base64.b64decode(packed)))
    runtime_assets = Path("/content/tattoogen-notebook-assets") / 'model_selection_prompts'
    runtime_assets.mkdir(parents=True, exist_ok=True)
    for name, content in assets.items():
        (runtime_assets / name).write_text(content, encoding="utf-8")
    RUNNER = runtime_assets / "ltx2_native.py"
    PROMPT_CSV = runtime_assets / 'model_selection_prompts.csv'
    print("Using the runner and prompt CSV bundled in this notebook.")
else:
    checkout = Path(SOURCE_REPO)
    RUNNER = checkout / "notebooks/ltx2_native.py"
    PROMPT_CSV = checkout / "data/metadata" / 'model_selection_prompts.csv'
    missing = [str(path) for path in (RUNNER, PROMPT_CSV) if not path.is_file()]
    if missing:
        raise FileNotFoundError("Fix SOURCE_REPO or set it to None. Missing: " + ", ".join(missing))
    print("Using the runner and prompts from", checkout)


## Preview the development jobs

In [ ]:
FRAME_COUNTS = [49, 65]
PROMPT_IDS = [1, 2, 3]
SEED = 42
RUN_GENERATION = False
CONTINUE_ON_ERROR = False

OUTPUT = RUNS / "ltx2_duration_test"
args = [
    str(RUNNER),
    "--phase", "duration_test", "--csv", str(PROMPT_CSV),
    "--output", str(OUTPUT), "--seed", str(SEED), "--frames", *map(str, FRAME_COUNTS),
]
args += ["--prompt-ids", *map(str, PROMPT_IDS)]

preview = subprocess.run([sys.executable, *args], text=True, capture_output=True)
if preview.returncode:
    raise RuntimeError(preview.stderr.strip() or preview.stdout.strip() or "Plan preview failed.")
plan = json.loads(preview.stdout)
print("GPU jobs:", len(plan["jobs"]), "| video frames:", sorted({j["frames"] for j in plan["jobs"]}))
print("Prompt IDs:", list(dict.fromkeys(j["prompt_id"] for j in plan["jobs"])))
print("Output:", OUTPUT)
print(json.dumps(plan["recipe"], indent=2))

## Install the pinned CUDA environment

Check the plan above before running this cell. This uses a separate Python 3.12 environment.

In [ ]:
capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True,
).splitlines()[0].strip()
if float(capability) < 8.9:
    raise RuntimeError(f"FP8 requires an L4 or newer supported GPU. Colab assigned SM {capability}.")
print("Free local disk GiB", round(shutil.disk_usage("/content").free / 2**30, 1))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
runtime = Path("/content/tattoogen-ltx2-env")
python = str(runtime / "bin/python")
if not Path(python).exists():
    subprocess.run([uv, "venv", "--python", "3.12", str(runtime)], check=True)
torch_packages = ["torch==2.9.1", "torchvision==0.24.1", "torchaudio==2.9.1"]
pip = [uv, "pip", "install", "--python", python]
subprocess.run([*pip, "--index-url", "https://download.pytorch.org/whl/cu128", *torch_packages], check=True)

revision = plan["recipe"]["code_revision"]
source = Path("/content") / ("ltx2-" + revision[:12])
if not (source / ".git").is_dir():
    subprocess.run(["git", "init", str(source)], check=True)
    subprocess.run(["git", "-C", str(source), "remote", "add", "origin",
                    "https://github.com/Lightricks/LTX-2.git"], check=True)
subprocess.run(["git", "-C", str(source), "fetch", "--depth", "1", "origin", revision], check=True)
subprocess.run(["git", "-C", str(source), "checkout", "--detach", revision], check=True)
constraints = runtime / "constraints.txt"
constraints.write_text(chr(10).join([*torch_packages, "transformers==5.14.1"]) + chr(10))
subprocess.run([
    *pip, "--no-sources", "-c", str(constraints),
    "-e", str(source / "packages/ltx-core"), "-e", str(source / "packages/ltx-pipelines"),
    "huggingface_hub",
], check=True)
subprocess.run([uv, "pip", "check", "--python", python], check=True)

## Generate and save

Set `RUN_GENERATION` to `True` in the plan cell and rerun that cell before starting.
Run one Colab session per output folder. Each completed MP4 and JSON is saved to Drive.
Rerun with the same settings to verify and skip completed jobs. Inspect errors before continuing.

In [ ]:
from google.colab import userdata

if not RUN_GENERATION:
    raise RuntimeError("Review the plan above and set RUN_GENERATION = True to start.")
# Keep credentials in the child process environment. Never print the environment.
worker_env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"), HF_HOME="/content/ltx2-hf")
# Use True if Xet stalls on this Colab VM.
USE_HTTP_DOWNLOAD = False
if USE_HTTP_DOWNLOAD:
    worker_env["HF_HUB_DISABLE_XET"] = "1"

OUTPUT.mkdir(parents=True, exist_ok=True)
with (OUTPUT / "runtime-packages.txt").open("a", encoding="utf-8") as packages:
    packages.write(chr(10))
    subprocess.run([uv, "pip", "freeze", "--python", python], stdout=packages, check=True)

command = [python, "-u", *args, "--run"]
if CONTINUE_ON_ERROR:
    command.append("--continue-on-error")
with subprocess.Popen(command, env=worker_env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as worker:
    try:
        for line in worker.stdout:
            print(line, end="", flush=True)
        if worker.wait():
            raise RuntimeError("Generation stopped. Inspect the output above and failures.jsonl, then rerun.")
    except KeyboardInterrupt:
        worker.terminate()
        raise

## Inspect the timings and choose a length

Copy the completed MP4 and JSON pairs to the evaluation machine and follow
`README_proposed.md` with `config/duration-test.json`. Compare accepted-pair yield,
reviewer-assessed variation, and total GPU inference minutes per accepted pair.
Use the timing table below for successful clips and failures.jsonl for failed attempts.
Preserve zero-yield clips and failed jobs in the report.
Record the chosen length and rationale before opening the baseline notebook.

In [ ]:
completed = []
for path in sorted(OUTPUT.glob("*.json")):
    if path.name == "plan.json":
        continue
    row = json.loads(path.read_text())
    completed.append({
        "prompt": row["prompt_id"], "frames": row["settings"]["actual_frames"],
        "seconds": round(row["settings"]["generation_and_export_seconds"], 1),
        "gpu": row["settings"]["gpu"], "sidecar": path.name,
    })
print(f"{len(completed)}/{len(plan['jobs'])} videos saved")
if completed:
    import pandas as pd
    display(pd.DataFrame(completed))
print("Drive output:", OUTPUT)